In [ ]:
import pandas as pd, pickle as pkl

In [ ]:
# ── CONFIG ─────────────────────────────────────────────────────────────────
# Set these to match config/paths.yaml
import os
PHENO_DIR   = "<set to cfg.gwas.pheno_dir in config/paths.yaml>"
FASTGWA_OUT = "<set to cfg.gwas.fastgwa_out in config/paths.yaml>"
# EXTERNAL inputs (not in config/paths.yaml):
EMBEDDING_DIR        = "<EXTERNAL: regional BRE embedding/representation directory>"
AE_INTERPRETATION_DIR = "<EXTERNAL: autoencoder interpretation pickle directory>"


In [ ]:
pl = [os.path.join(PHENO_DIR, "T1_pheno_discovery"),
      os.path.join(PHENO_DIR, "T1_pheno_replication"),
      os.path.join(PHENO_DIR, "T2_pheno_discovery"),
      os.path.join(PHENO_DIR, "T2_pheno_replication")]

In [ ]:
def break_pheno(x):
    df = pd.read_table(x, sep=' ').drop_duplicates(subset='IID')
    for i in range(len(df.columns)-2):
        df[["FID", "IID", f"QT{i}"]].to_csv(x+f"QT{i}", sep=' ', index=False)

In [ ]:
for p in pl:
    break_pheno(p)

In [ ]:
def proc(x, dim, cohort):
    df = pd.DataFrame.from_dict(x, orient='index')
    for j in range(dim):
        df[f'QT{j}'] = df[1].apply(lambda x: x[j])
    df['FID'] = df.index
    df['IID'] = df.index
    for j in range(dim):
        df[["FID", "IID", f"QT{j}"]].to_csv(os.path.join(PHENO_DIR, f"model{dim}", f"T1_pheno_{cohort}QT{j}"), sep=' ', index=False)

In [ ]:
for dim in [32, 64]:
    for cohort in ['discovery', 'replication']: 
        d = pkl.load(open(os.path.join(AE_INTERPRETATION_DIR, f"T1_{dim}_interpretation", f"dict_T1_{dim}_{cohort}.pkl"), 'rb'))
        proc(d, dim, cohort)


# This is my edit. Everything above this cell is from the upstream covariate/pheno-prep code. Do not edit above.

In [ ]:
import os
import glob
import pandas as pd

In [ ]:
def break_pheno(x,fastgwa_folder):
    print(f"Working with {x}")
    region_name = os.path.basename(x)
    df = pd.read_csv(x,sep=',').drop_duplicates(subset='IID')
    for i in range(2,df.shape[1]):
        df[["FID", "IID", f"QT{i-2}"]].to_csv(os.path.join(fastgwa_folder,region_name+f"QT{i-2}"), sep=' ', index=False)


In [ ]:
region_dir_path = EMBEDDING_DIR

In [ ]:

pheno_file_pattern = '[0-9]*_*mean.csv'
pattern = os.path.join(region_dir_path,'**',pheno_file_pattern)
pheon_files = glob.glob(pattern)

In [ ]:

fastgwa_folder = os.path.join(region_dir_path,'fastgws')
os.makedirs(fastgwa_folder,exist_ok=True)

In [ ]:
for p in pheon_files:
    break_pheno(p,fastgwa_folder)
    

In [ ]:
len(os.listdir(fastgwa_folder))

In [ ]:
128*18

In [ ]:
fastgwa_folder

In [ ]:
import glob
import os
import shutil
discovery_dir = os.path.join(PHENO_DIR, '16_Brain_Stem_or_4th_Ventricle_mean**')
brain_stems = glob.glob(discovery_dir)

In [ ]:
brain_stems

In [ ]:
for file in brain_stems:
    os.rename(file, file.replace(" ",""))

In [ ]:
fastgwa_pheno_dir= PHENO_DIR

l = sorted(glob.glob(f"{fastgwa_pheno_dir}/16_Brain_Stem_or_4th_Ventricle_mean**"))

In [ ]:
l

In [ ]:
glob.glob(f"{fastgwa_pheno_dir}/16_Brain_Stem_or_4th_Ventricle_mean**")

In [ ]:
fastgwa_pheno_dir= PHENO_DIR

l = sorted(glob.glob(f"{fastgwa_pheno_dir}/16_Brain_Stem_or_4th_Ventricle_mean**"))
num_run = len(l)

In [ ]:
num_run

In [ ]:
from matplotlib import pyplot as plt
import pandas as pd
import os
import numpy as np
from tqdm import tqdm
from multiprocessing import Pool
from itertools import zip_longest
from subprocess import check_output, STDOUT
from glob import glob
import re
import pickle as pkl

In [ ]:
def convert2float(x):
    try:
        return float(x)
    except:
        return 1.0

def extract_col(x, offset=0):
    out = check_output(f"awk '{{print $(NF-{offset})}}' '{x}'", universal_newlines=True, shell=True, stderr=STDOUT)
    return np.array(list(map(convert2float, out.strip('\n').split('\n')[1:]))), x

def create_minP(glob_list, pcol=0, mode='min'):
    # pcol is indexed from right to left, 0 means last col
    if mode == 'min':
        op = np.argmin
    elif mode == 'max':
        op == np.argmax
    else:
        raise Exception('not implemented')
    batch = 50
    for i in tqdm(range(0, len(glob_list), batch)):
        with Pool(batch) as q:
            result = q.starmap(extract_col, zip_longest(glob_list[i:i+batch], (), fillvalue=pcol))
        pnew, fnew = list(zip(*result))
        if i == 0:
            pnew = np.vstack(pnew) # (batch,n_snps)
            idx = op(pnew, 0)# (n_snps), what is the minimum p for each snp
            f = np.array(fnew, dtype=object)[idx] # retaining only minum p-val file for each snp
        else:
            pnew = list(pnew)
            pnew.append(p)
            pnew = np.vstack(pnew)
            idx = op(pnew, 0)
            mask = (idx != (pnew.shape[0]-1))
            f[mask] = np.array(fnew, dtype=object)[idx[mask]]
        p = pnew[idx, np.arange(pnew.shape[1])] # here i am selecting the row where minum p-value for each snp was found (n_batch, n_snp)
    return p, f

In [ ]:
sum_stat_files_dir = FASTGWA_OUT

In [ ]:
sum_files = glob(f"{sum_stat_files_dir}/discovery_*.fastGWA")

In [ ]:
wmc_regions = [ '1_CSF_mean',
 '2_GM_mean',
 '3_WM_mean']
sub_cortical_sum_stas = []

In [ ]:
sub_wmc_sumstats = [p for p in sum_files if (len(re.findall(re.compile(wmc_regions[0]),p))>0 or   len(re.findall(re.compile(wmc_regions[1]),p))>0 or len(re.findall(re.compile(wmc_regions[2]),p))>0   )]

In [ ]:
sub_cort_sumstats = [p for p in sum_files if (len(re.findall(re.compile(wmc_regions[0]),p))==0 and   len(re.findall(re.compile(wmc_regions[1]),p))==0 and len(re.findall(re.compile(wmc_regions[2]),p))==0   )]

In [ ]:
#p_subc,f_subc = create_minP(sub_cort_sumstats, pcol=1, mode='min')

In [ ]:
batch = 50
glob_list = sub_cort_sumstats
pcol=1
total_signif_pairs =0
for i in tqdm(range(0, len(glob_list), batch)):
    with Pool(batch) as q:
        result = q.starmap(extract_col, zip_longest(glob_list[i:i+batch], (), fillvalue=pcol))
    pnew, fnew = list(zip(*result))
    t = np.vstack(pnew)
    total_signif_pairs += (t<2.6e-11).sum()

In [ ]:
total_signif_pairs

In [ ]:
t.shape

In [ ]:
batch = 50
glob_list = sub_cort_sumstats
pcol=1
total_signif_pairs =0
for i in tqdm(range(0, len(glob_list), batch)):
    with Pool(batch) as q:
        result = q.starmap(extract_col, zip_longest(glob_list[i:i+batch], (), fillvalue=pcol))
    pnew, fnew = list(zip(*result))
    t = np.vstack(pnew)
    total_signif_pairs += (t<2.6e-11).sum()

In [ ]:
pnew, fnew = list(zip(*result))

In [ ]:
if i == 0:
    pnew = np.vstack(pnew)

In [ ]:
i

In [ ]:
pnew[0].shape

In [ ]:
t = np.vstack(pnew)
t.shape

In [ ]:
(t<2.6e-11).sum()

### Average Sub-Cortical phenotypes

In [ ]:
def break_pheno(x,fastgwa_folder):
    print(f"Working with {x}")
    region_name = os.path.basename(x)
    df = pd.read_csv(x,sep=' ').drop_duplicates(subset='IID')
    for i in range(2,df.shape[1]):
        df[["FID", "IID", f"QT{i-2}"]].to_csv(os.path.join(fastgwa_folder,region_name+f"QT{i-2}"), sep=' ', index=False)


In [ ]:
region_dir_path = EMBEDDING_DIR
discovery_file = os.path.join(region_dir_path,'discovery.csv')
replication_file = os.path.join(region_dir_path,'replication.csv')

fastgwa_folder_discovery = os.path.join(region_dir_path,'fastgws/discovery')
os.makedirs(fastgwa_folder_discovery,exist_ok=True)

fastgwa_folder_replication = os.path.join(region_dir_path,'fastgws/replication')
os.makedirs(fastgwa_folder_replication,exist_ok=True)



In [ ]:
break_pheno(discovery_file,fastgwa_folder_discovery)

In [ ]:
break_pheno(replication_file,fastgwa_folder_replication)

In [ ]:
ds = pd.read_csv(os.path.join(EMBEDDING_DIR, "fastgws/discovery/discovery.csvQT121"), sep=" ")

In [ ]:
ds.shape